# Dự báo Doanh thu Sản phẩm
## 6. Huấn luyện Mô hình & Benchmarking

Chúng ta tiến hành huấn luyện 4 cấu trúc Gradient Boosting chính thức song song với mô hình tự cài đặt **LightGBM tự build từ đầu**:
- Scikit-learn HistGradientBoostingRegressor
- XGBoost Regressor
- LightGBM Regressor
- CatBoost Regressor
- **LightGBM from scratch** (định nghĩa chi tiết bên dưới)

Mỗi mô hình sẽ được ghi nhận thời gian huấn luyện thực tế.

In [ ]:
import pandas as pd
import joblib
import time
import os
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

READY_DATA_DIR = r'../data/ready_for_train'
X_train = joblib.load(os.path.join(READY_DATA_DIR, 'X_train.pkl'))
y_train = joblib.load(os.path.join(READY_DATA_DIR, 'y_train.pkl'))

print('Kích thước tập huấn luyện:', X_train.shape)

### 6.1 Mô hình LightGBM tự build từ đầu (LightGBM from scratch)

Mô hình tự thiết lập bao gồm các cơ chế lõi của LightGBM:
1. **Histogram-based Binning:** Chia các đặc trưng liên tục vào các khoảng Histogram cố định để tăng tốc độ quét điểm cắt.
2. **Leaf-wise tree growth:** Phát triển cây theo chiều sâu node tốt nhất thay vì theo cấp độ của tầng.
3. **Gradient Boosting loop:** Fit chuỗi cây hồi quy liên tục trên các sai số residual.

In [ ]:
import sys

class LightGBMFromScratch:
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3, max_leaf_nodes=8, n_bins=32):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.max_leaf_nodes = max_leaf_nodes
        self.n_bins = n_bins
        self.base_pred = None
        self.trees = []
        self.bin_edges = {}

    def _bin_features(self, X, fit=False):
        X_binned = X.copy()
        for col in X.columns:
            if fit:
                n_unique = X[col].nunique()
                if n_unique > self.n_bins:
                    percentiles = np.linspace(0, 100, self.n_bins + 1)
                    edges = np.percentile(X[col], percentiles)
                    edges = np.unique(edges)
                    self.bin_edges[col] = edges
                else:
                    self.bin_edges[col] = None
            edges = self.bin_edges[col]
            if edges is not None:
                X_binned[col] = np.digitize(X[col], edges[1:-1])
        return X_binned

    def fit(self, X, y):
        X_binned = self._bin_features(X, fit=True)
        self.base_pred = np.mean(y)
        f_m = np.full(len(y), self.base_pred)
        
        from sklearn.tree import DecisionTreeRegressor
        self.trees = []
        for i in range(self.n_estimators):
            gradient = y - f_m
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                max_leaf_nodes=self.max_leaf_nodes,
                random_state=42 + i
            )
            tree.fit(X_binned, gradient)
            f_m += self.learning_rate * tree.predict(X_binned)
            self.trees.append(tree)

    def predict(self, X):
        X_binned = self._bin_features(X, fit=False)
        preds = np.full(len(X), self.base_pred)
        for tree in self.trees:
            preds += self.learning_rate * tree.predict(X_binned)
        return preds

# Đăng ký lớp vào module __main__ để pickle tải mô hình thuận tiện
sys.modules['__main__'].LightGBMFromScratch = LightGBMFromScratch

### 6.2 Khởi tạo danh sách mô hình huấn luyện

In [ ]:
models = {
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, random_seed=42, verbose=0, allow_writing_files=False),
    'LightGBM (Scratch)': LightGBMFromScratch(n_estimators=50, learning_rate=0.1, max_depth=5, max_leaf_nodes=15, n_bins=32)
}

### 6.3 Tiến hành huấn luyện mô hình

In [ ]:
trained_models = {}
fit_times = {}

for name, model in models.items():
    print(f'Đang huấn luyện mô hình: {name}...')
    start_time = time.time()
    model.fit(X_train, y_train)
    fit_times[name] = time.time() - start_time
    trained_models[name] = model
    print(f'Hoàn thành huấn luyện {name} trong {fit_times[name]:.2f} giây.')

### 6.4 Lưu trữ trọng số mô hình đã huấn luyện

In [ ]:
model_dir = r'../data/ready_for_train'

for name, model in trained_models.items():
    clean_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    model_path = os.path.join(model_dir, f'model_{clean_name}.pkl')
    joblib.dump(model, model_path)
    print(f'Đã lưu mô hình {name} tại {model_path}')

joblib.dump(fit_times, os.path.join(model_dir, 'fit_times.pkl'))